# Keyword and Entity Extraction using IBM WatsonX.ai

This notebook demonstrates how to extract keywords and entities from text data using IBM WatsonX.ai foundation models.

## Table of Contents
1. [Setup and Installation](#setup)
2. [Load Dataset](#load-data)
3. [Initialize WatsonX.ai Client](#init-watsonx)
4. [Keyword Extraction](#keyword-extraction)
5. [Entity Extraction](#entity-extraction)
6. [Batch Processing](#batch-processing)
7. [Save Results](#save-results)

## 1. Setup and Installation <a name="setup"></a>

First, install the required packages.

In [2]:
# Install required packages
#pip install ibm-watsonx-ai>=1.1.11 pandas python-dotenv ibm-cos-sdk

### Import Required Libraries

In [1]:
import os
import pandas as pd
import json
from typing import List, Dict, Optional
from dotenv import load_dotenv

# IBM WatsonX.ai imports
from ibm_watsonx_ai.credentials import Credentials
from ibm_watsonx_ai import APIClient
from ibm_watsonx_ai.foundation_models import ModelInference

# IBM Cloud Object Storage imports (optional)
import ibm_boto3
from botocore.client import Config
import types

print("✓ All libraries imported successfully")

✓ All libraries imported successfully


### Configure Credentials

Set up your IBM Cloud credentials. You can either:
- Use environment variables (recommended)
- Set them directly in the notebook (not recommended for production)

**Required Credentials:**
- `WATSONX_API_KEY`: Your IBM Cloud API key
- `WATSONX_PROJECT_ID`: Your WatsonX.ai project ID

**Optional (for COS):**
- `COS_API_KEY`: Cloud Object Storage API key
- `COS_ENDPOINT`: COS endpoint URL
- `COS_BUCKET`: COS bucket name

Refer to `IBM_CREDENTIALS_SETUP.md` for detailed instructions on obtaining these credentials.

In [4]:
# Load environment variables from .env file (if exists)
load_dotenv()

# Option 1: Use environment variables (recommended)
WATSONX_API_KEY = os.getenv("WATSONX_API_KEY")
WATSONX_PROJECT_ID = os.getenv("WATSONX_PROJECT_ID")
WATSONX_URL = os.getenv("WATSONX_URL", "https://us-south.ml.cloud.ibm.com")

# Option 2: Set directly (not recommended for production)
# WATSONX_API_KEY = "your-api-key-here"
# WATSONX_PROJECT_ID = "your-project-id-here"
# WATSONX_URL = "https://us-south.ml.cloud.ibm.com"

# Validate credentials
if not WATSONX_API_KEY or not WATSONX_PROJECT_ID:
    raise ValueError("❌ WATSONX_API_KEY and WATSONX_PROJECT_ID must be set")

print("✓ Credentials loaded successfully")
print(f"  Project ID: {WATSONX_PROJECT_ID[:8]}...")
print(f"  WatsonX URL: {WATSONX_URL}")

✓ Credentials loaded successfully
  Project ID: 190564dc...
  WatsonX URL: https://us-south.ml.cloud.ibm.com


## 2. Load Dataset <a name="load-data"></a>

Load the dataset from either:
- **Local file**: CSV file in the data folder
- **IBM Cloud Object Storage (COS)**: For cloud-based storage

The dataset should have at least these columns:
- `url`: The webpage URL
- `text`: The text content to analyze

### Option A: Load from Local File

In [5]:
# Load from local CSV file
DATA_PATH = "../data/with_label.csv"

try:
    df = pd.read_csv(DATA_PATH)
    print(f"✓ Dataset loaded successfully from {DATA_PATH}")
    print(f"  Shape: {df.shape}")
    print(f"  Columns: {list(df.columns)}")
    
    # Display first few rows
    print("\nFirst 3 rows:")
    display(df.head(3))
except FileNotFoundError:
    print(f"❌ File not found: {DATA_PATH}")
    print("Please ensure the file exists or use COS option below")

✓ Dataset loaded successfully from ../data/with_label.csv
  Shape: (1153, 3)
  Columns: ['url', 'text', 'label']

First 3 rows:


,url,text,label
0,https://abcnews.go.com/International/photos/is...,Israel-Gaza-Lebanon-Syria conflict: Slideshow ...,['/arts_&_entertainment/online_media/online_im...
1,https://acapt.org/educationevents/advancing-ac...,Advancing Accessibility & Disability Equity Su...,['/business_&_industrial/business_services/cor...
2,https://adcable.com/products/rf-communication....,RF Wire & Cable Manufacturer | Advanced Digita...,['/computers_&_electronics/electronics_&_elect...


### Option B: Load from IBM Cloud Object Storage (COS)

In [7]:
# COS Configuration
USE_COS = True  # Set to True to use COS

if USE_COS:
    # COS Credentials
    COS_API_KEY = os.getenv("COS_API_KEY")
    COS_ENDPOINT = os.getenv("COS_ENDPOINT", "https://s3.us-south.cloud-object-storage.appdomain.cloud")
    COS_BUCKET = os.getenv("COS_BUCKET")
    COS_OBJECT_KEY = os.getenv("COS_OBJECT_KEY", "with_label.csv")
    
    # Validate COS credentials
    if not all([COS_API_KEY, COS_ENDPOINT, COS_BUCKET]):
        raise ValueError("❌ COS_API_KEY, COS_ENDPOINT, and COS_BUCKET must be set when USE_COS=True")
    
    # Initialize COS client
    cos_client = ibm_boto3.client(
        service_name='s3',
        ibm_api_key_id=COS_API_KEY,
        ibm_auth_endpoint="https://iam.cloud.ibm.com/identity/token",
        config=Config(signature_version='oauth'),
        endpoint_url=COS_ENDPOINT
    )
    
    # Read CSV from COS
    body = cos_client.get_object(Bucket=COS_BUCKET, Key=COS_OBJECT_KEY)['Body']
    
    # Add missing __iter__ method for pandas compatibility
    if not hasattr(body, "__iter__"):
        def __iter__(self):
            return 0
        body.__iter__ = types.MethodType(__iter__, body)
    
    df = pd.read_csv(body)
    print(f"✓ Dataset loaded successfully from COS")
    print(f"  Bucket: {COS_BUCKET}")
    print(f"  Object: {COS_OBJECT_KEY}")
    print(f"  Shape: {df.shape}")
    print(f"  Columns: {list(df.columns)}")
    
    # Display first few rows
    print("\nFirst 3 rows:")
    display(df.head(3))
else:
    print("ℹ️  COS loading is disabled. Using local file option.")

✓ Dataset loaded successfully from COS
  Bucket: multilabelclassification-donotdelete-pr-x5pzbfaipzelvs
  Object: with_label.csv
  Shape: (1153, 3)
  Columns: ['url', 'text', 'label']

First 3 rows:


,url,text,label
0,https://abcnews.go.com/International/photos/is...,Israel-Gaza-Lebanon-Syria conflict: Slideshow ...,['/arts_&_entertainment/online_media/online_im...
1,https://acapt.org/educationevents/advancing-ac...,Advancing Accessibility & Disability Equity Su...,['/business_&_industrial/business_services/cor...
2,https://adcable.com/products/rf-communication....,RF Wire & Cable Manufacturer | Advanced Digita...,['/computers_&_electronics/electronics_&_elect...


### Prepare Data for Processing

In [9]:
# Select a sample for demonstration (remove this for full processing)
SAMPLE_SIZE = 5  # Process first 5 rows for demo
df_sample = df.head(SAMPLE_SIZE).copy()

print(f"✓ Prepared {len(df_sample)} samples for processing")
print(f"\nSample data:")
display(df_sample[['url', 'text']].head())

✓ Prepared 5 samples for processing

Sample data:


,url,text
0,https://abcnews.go.com/International/photos/is...,Israel-Gaza-Lebanon-Syria conflict: Slideshow ...
1,https://acapt.org/educationevents/advancing-ac...,Advancing Accessibility & Disability Equity Su...
2,https://adcable.com/products/rf-communication....,RF Wire & Cable Manufacturer | Advanced Digita...
3,https://adventuregamers.com/companies/view/27602,Adventure games by Terrible Toybox | Adventure...
4,https://aggie-horticulture.tamu.edu/vegetable/...,Ginger - Vegetable Resources Vegetable Resourc...


## 3. Initialize WatsonX.ai Client <a name="init-watsonx"></a>

Initialize the WatsonX.ai foundation model for keyword and entity extraction.

In [10]:
# Initialize WatsonX.ai credentials
credentials = Credentials(
    url=WATSONX_URL,
    api_key=WATSONX_API_KEY
)

# Model configuration for keyword/entity extraction
# Using Mistral model (similar to categorization app)
MODEL_ID = "mistralai/mistral-small-3-1-24b-instruct-2503"

# Alternative models you can try:
# MODEL_ID = "meta-llama/llama-3-1-70b-instruct"
# MODEL_ID = "ibm/granite-13b-chat-v2"

parameters = {
    "decoding_method": "greedy",  # Use greedy for more deterministic results
    "temperature": 0.1,  # Low temperature for consistent extraction
    "max_new_tokens": 500,  # Enough tokens for keywords and entities
    "min_new_tokens": 10,
    "repetition_penalty": 1.0,
    "stop_sequences": ["\n\n\n"]  # Stop at triple newline
}

# Initialize model
model = ModelInference(
    model_id=MODEL_ID,
    params=parameters,
    credentials=credentials,
    project_id=WATSONX_PROJECT_ID
)

print("✓ WatsonX.ai model initialized successfully")
print(f"  Model: {MODEL_ID}")
print(f"  Temperature: {parameters['temperature']}")
print(f"  Max tokens: {parameters['max_new_tokens']}")

✓ WatsonX.ai model initialized successfully
  Model: mistralai/mistral-small-3-1-24b-instruct-2503
  Temperature: 0.1
  Max tokens: 500


## 4. Keyword Extraction <a name="keyword-extraction"></a>

Extract important keywords from the text content.

### Define Keyword Extraction Function

In [15]:
def extract_keywords(text: str, url: str = "", max_keywords: int = 10) -> List[str]:
    """
    Extract keywords and phrases from text using WatsonX.ai
    
    Args:
        text: Text content to analyze
        url: Optional URL for context
        max_keywords: Maximum number of keywords and phrases to extract
    
    Returns:
        List of extracted keywords and phrases
    """
    prompt = f"""You are an expert at extracting important keywords and phrases from text content.

Task: Extract the {max_keywords} most important keywords or key phrases from the following text.

Instructions:
- Focus on nouns, noun phrases, and important concepts
- Include proper nouns (names, places, organizations)
- Avoid common stop words
- Return keywords as a JSON array of strings
- If you find a keyword or phrase that is a sub-phrase of another keyword or phrase, ignore it
- Do not include repeated keywords or phrases
- Do not include keywords or phrases that are sub-phrases of other keywords or phrases
- if you don't found max_keywords, return less keywords and phrases but don't include repeated keywords or phraase in the result.

URL: {url}

Text:
{text}

Keywords (JSON array):"""
    
    try:
        result = model.generate_text(prompt=prompt, guardrails=False)
        
        # Parse JSON result
        result = result.strip()
        if result.startswith('['):
            keywords = json.loads(result)
            return keywords if isinstance(keywords, list) else []
        else:
            # Fallback: split by comma or newline
            keywords = [k.strip().strip('"').strip("'") for k in result.replace('\n', ',').split(',')]
            return [k for k in keywords if k][:max_keywords]
    except Exception as e:
        print(f"Error extracting keywords: {e}")
        return []

print("✓ Keyword extraction function defined")

✓ Keyword extraction function defined


### Test Keyword Extraction on Sample

In [16]:
# Test on first sample
sample_text = df_sample.iloc[0]['text']
sample_url = df_sample.iloc[0]['url']

print("Testing keyword extraction...\n")
print(f"URL: {sample_url}")
print(f"\nText (first 200 chars): {sample_text}...\n")

keywords = extract_keywords(sample_text, sample_url, max_keywords=20)

print(f"\n✓ Extracted {len(keywords)} keywords:")
for i, keyword in enumerate(keywords, 1):
    print(f"  {i}. {keyword}")

Testing keyword extraction...

URL: https://abcnews.go.com/International/photos/israel-gaza-conflict-slideshow-103808214

Text (first 200 chars): Israel-Gaza-Lebanon-Syria conflict: Slideshow - ABC News, Photos of the Israel-Gaza-Lebanon-Syria conflict., Israel, Palestine, Israeli, Palestinian, Gaza, Gaza Strip, Hamas, Hezbollah, Lebanon, rockets, missiles, Middle East, Tel Aviv, Syria, news slideshow, Slideshow, 103808214...


✓ Extracted 13 keywords:
  1. Israel-Gaza-Lebanon-Syria conflict
  2. Israel
  3. Palestine
  4. Gaza Strip
  5. Hamas
  6. Hezbollah
  7. Lebanon
  8. Middle East
  9. Tel Aviv
  10. Syria
  11. rockets
  12. missiles
  13. news slideshow


## 5. Entity Extraction <a name="entity-extraction"></a>

Extract named entities (people, organizations, locations, etc.) from the text.

### Define Entity Extraction Function

In [30]:
import re


def extract_entities(text: str, url: str = "") -> Dict[str, List[str]]:
    """
    Extract named entities from text using WatsonX.ai
    
    Args:
        text: Text content to analyze
        url: Optional URL for context
    
    Returns:
        Dictionary with entity types as keys and lists of entities as values
    """
    prompt = f"""You are an expert at extracting named entities from text content.

Task: Extract named entities from the following text and categorize them.

Entity Categories:
- PERSON: Names of people
- ORGANIZATION: Companies, institutions, organizations
- LOCATION: Cities, countries, regions, places
- DATE: Dates, times, periods
- EVENT: Named events, conferences, incidents
- PRODUCT: Products, services, technologies
- OTHER: Other important entities

Instructions:
- Extract all relevant named entities
- Return as a JSON object with entity types as keys
- Each key should have an array of entity strings
- Remove duplicates
- Return only unique entities
- if you don't find any enetites then retrun 'No enities found'
- provide a valid JSON object don't include any explanation and other text

URL: {url}

Text:
{text}

Entities (JSON object):"""
    
    try:
        result = model.generate_text(prompt=prompt, guardrails=False)
        print(("Result:", result))
        
        # Parse JSON result - handle extra text before/after JSON
        result = result.strip()
        
        # Find the first '{' and last '}' to extract JSON
        start_idx = result.find('{')
        end_idx = result.rfind('}')
        
        if start_idx != -1 and end_idx != -1 and end_idx > start_idx:
            # Extract only the JSON portion
            json_str = result[start_idx:end_idx+1]
            
            try:
                entities = json.loads(json_str)
                return entities if isinstance(entities, dict) else {}
            except json.JSONDecodeError as json_err:
                print(f"JSON decode error: {json_err}")
                print(f"Attempted to parse: {json_str[:200]}...")
                return {}
        else:
            print("No valid JSON object found in response")
            return {}
        
    except Exception as e:
        print(f"Error extracting entities: {e}")
        return {}


print("✓ Entity extraction function defined")

✓ Entity extraction function defined


### Test Entity Extraction on Sample

In [32]:
# Test on first sample
print("Testing entity extraction...\n")
print(f"URL: {sample_url}")
print(f"\nText (first 200 chars): {df_sample.iloc[0]['text']}...\n")

entities = extract_entities(df_sample.iloc[0]['text'], url=df_sample.iloc[0]['url'])

print(f"\n✓ Extracted entities:",)
for entity_type, entity_list in entities.items():
    if entity_list:
        print(f"\n  {entity_type}:")
        for entity in entity_list:
            print(f"    - {entity}")

Testing entity extraction...

URL: https://abcnews.go.com/International/photos/israel-gaza-conflict-slideshow-103808214

Text (first 200 chars): Israel-Gaza-Lebanon-Syria conflict: Slideshow - ABC News, Photos of the Israel-Gaza-Lebanon-Syria conflict., Israel, Palestine, Israeli, Palestinian, Gaza, Gaza Strip, Hamas, Hezbollah, Lebanon, rockets, missiles, Middle East, Tel Aviv, Syria, news slideshow, Slideshow, 103808214...

('Result:', ' {\n  "PERSON": [],\n  "ORGANIZATION": [\n    "Hamas",\n    "Hezbollah"\n  ],\n  "LOCATION": [\n    "Israel",\n    "Palestine",\n    "Gaza",\n    "Gaza Strip",\n    "Lebanon",\n    "Tel Aviv",\n    "Syria",\n    "Middle East"\n  ],\n  "DATE": [],\n  "EVENT": [\n    "Israel-Gaza-Lebanon-Syria conflict",\n    "news slideshow"\n  ],\n  "PRODUCT": [],\n  "OTHER": [\n    "rockets",\n    "missiles"\n  ]\n}')

✓ Extracted entities:

  ORGANIZATION:
    - Hamas
    - Hezbollah

  LOCATION:
    - Israel
    - Palestine
    - Gaza
    - Gaza Strip
    - Lebanon

## 6. Batch Processing <a name="batch-processing"></a>

Process all samples in the dataset to extract keywords and entities.

In [33]:
def process_batch(df: pd.DataFrame, max_keywords: int = 10) -> pd.DataFrame:
    """
    Process a batch of texts to extract keywords and entities
    
    Args:
        df: DataFrame with 'url' and 'text_truncated' columns
        max_keywords: Maximum number of keywords to extract per text
    
    Returns:
        DataFrame with added 'keywords' and 'entities' columns
    """
    results = []
    
    for idx, row in df.iterrows():
        print(f"\nProcessing {idx + 1}/{len(df)}: {row['url'][:60]}...")
        
        try:
            # Extract keywords
            keywords = extract_keywords(
                text=row['text'],
                url=row['url'],
                max_keywords=max_keywords
            )
            print(f"  ✓ Keywords: {len(keywords)} extracted")
            
            # Extract entities
            entities = extract_entities(
                text=row['text'],
                url=row['url']
            )
            total_entities = sum(len(v) for v in entities.values())
            print(f"  ✓ Entities: {total_entities} extracted")
            
            results.append({
                'keywords': keywords,
                'entities': entities
            })
            
        except Exception as e:
            print(f"  ❌ Error: {e}")
            results.append({
                'keywords': [],
                'entities': {}
            })
    
    # Add results to dataframe
    df['keywords'] = [r['keywords'] for r in results]
    df['entities'] = [r['entities'] for r in results]
    
    return df

print("✓ Batch processing function defined")

✓ Batch processing function defined


### Run Batch Processing

In [34]:
# Process the sample dataset
print(f"Starting batch processing of {len(df_sample)} samples...\n")
print("=" * 70)

df_results = process_batch(df_sample, max_keywords=10)

print("\n" + "=" * 70)
print("\n✓ Batch processing complete!")
print(f"  Processed: {len(df_results)} samples")
print(f"  Columns: {list(df_results.columns)}")

Starting batch processing of 5 samples...


Processing 1/5: https://abcnews.go.com/International/photos/israel-gaza-conf...
  ✓ Keywords: 10 extracted
('Result:', ' {\n  "PERSON": [],\n  "ORGANIZATION": [\n    "Hamas",\n    "Hezbollah"\n  ],\n  "LOCATION": [\n    "Israel",\n    "Palestine",\n    "Gaza",\n    "Gaza Strip",\n    "Lebanon",\n    "Tel Aviv",\n    "Syria",\n    "Middle East"\n  ],\n  "DATE": [],\n  "EVENT": [\n    "Israel-Gaza-Lebanon-Syria conflict",\n    "news slideshow"\n  ],\n  "PRODUCT": [],\n  "OTHER": [\n    "rockets",\n    "missiles"\n  ]\n}')
  ✓ Entities: 14 extracted

Processing 2/5: https://acapt.org/educationevents/advancing-accessibility---...
  ✓ Keywords: 10 extracted
('Result:', ' {\n  "PERSON": [\n    "Lisa Meeks",\n    "Amanda Sharp"\n  ],\n  "ORGANIZATION": [\n    "The Inclusive Horizons Summit",\n    "Shirley Ryan AbilityLab",\n    "Northwestern University Physical Therapy and Human Movement Sciences",\n    "Federation of State Boards of Physical Therap

### View Results

In [36]:
# Display results
print("\nExtraction Results:\n")

for idx, row in df_results.iterrows():
    print(f"\n{'='*70}")
    print(f"Sample {idx + 1}")
    print(f"{'='*70}")
    print(f"\nURL: {row['url']}")
    print(f"\nText (first 150 chars): {row['text'][:150]}...")
    
    print(f"\nKeywords ({len(row['keywords'])}):")  
    for i, kw in enumerate(row['keywords'], 1):
        print(f"  {i}. {kw}")
    
    print(f"\nEntities:")
    if row['entities']:
        for entity_type, entity_list in row['entities'].items():
            if entity_list:
                print(f"  {entity_type}: {', '.join(entity_list)}")
    else:
        print("  (none extracted)")


Extraction Results:


Sample 1

URL: https://abcnews.go.com/International/photos/israel-gaza-conflict-slideshow-103808214

Text (first 150 chars): Israel-Gaza-Lebanon-Syria conflict: Slideshow - ABC News, Photos of the Israel-Gaza-Lebanon-Syria conflict., Israel, Palestine, Israeli, Palestinian, ...

Keywords (10):
  1. Israel-Gaza-Lebanon-Syria conflict
  2. Israel
  3. Palestine
  4. Gaza Strip
  5. Hamas
  6. Hezbollah
  7. Lebanon
  8. Middle East
  9. Tel Aviv
  10. Syria

Entities:
  ORGANIZATION: Hamas, Hezbollah
  LOCATION: Israel, Palestine, Gaza, Gaza Strip, Lebanon, Tel Aviv, Syria, Middle East
  EVENT: Israel-Gaza-Lebanon-Syria conflict, news slideshow
  OTHER: rockets, missiles

Sample 2

URL: https://acapt.org/educationevents/advancing-accessibility---disability-summit

Text (first 150 chars): Advancing Accessibility & Disability Equity Summit, The Inclusive Horizons Summit was an in-person, one-and-a-half day summit at the Shirley Ryan Abil...

Keywords (10):
  1. physi

## 7. Save Results <a name="save-results"></a>

Save the extracted keywords and entities to a CSV file.

In [37]:
# Prepare data for export
df_export = df_results.copy()

# Convert lists and dicts to strings for CSV export
df_export['keywords'] = df_export['keywords'].apply(lambda x: json.dumps(x))
df_export['entities'] = df_export['entities'].apply(lambda x: json.dumps(x))

# Save to CSV
output_path = "../data/extraction_results.csv"
df_export.to_csv(output_path, index=False)

print(f"✓ Results saved to: {output_path}")
print(f"  Rows: {len(df_export)}")
print(f"  Columns: {list(df_export.columns)}")

✓ Results saved to: ../data/extraction_results.csv
  Rows: 5
  Columns: ['url', 'text', 'label', 'keywords', 'entities']


### Optional: Save to JSON for Better Structure

In [39]:
# Save as JSON for better structure preservation
json_output_path = "../data/extraction_results.json"

# Convert to list of dictionaries
results_json = df_results[['url', 'text', 'keywords', 'entities']].to_dict('records')

with open(json_output_path, 'w', encoding='utf-8') as f:
    json.dump(results_json, f, indent=2, ensure_ascii=False)

print(f"✓ Results also saved to: {json_output_path}")

✓ Results also saved to: ../data/extraction_results.json


## Summary Statistics

In [40]:
# Calculate statistics
total_keywords = sum(len(kw) for kw in df_results['keywords'])
avg_keywords = total_keywords / len(df_results) if len(df_results) > 0 else 0

total_entities = sum(sum(len(v) for v in ent.values()) for ent in df_results['entities'])
avg_entities = total_entities / len(df_results) if len(df_results) > 0 else 0

print("\n" + "="*70)
print("EXTRACTION SUMMARY")
print("="*70)
print(f"\nTotal samples processed: {len(df_results)}")
print(f"\nKeywords:")
print(f"  Total extracted: {total_keywords}")
print(f"  Average per sample: {avg_keywords:.2f}")
print(f"\nEntities:")
print(f"  Total extracted: {total_entities}")
print(f"  Average per sample: {avg_entities:.2f}")
print(f"\nOutput files:")
print(f"  CSV: {output_path}")
print(f"  JSON: {json_output_path}")
print("\n" + "="*70)


EXTRACTION SUMMARY

Total samples processed: 5

Keywords:
  Total extracted: 47
  Average per sample: 9.40

Entities:
  Total extracted: 66
  Average per sample: 13.20

Output files:
  CSV: ../data/extraction_results.csv
  JSON: ../data/extraction_results.json



## Next Steps

1. **Process Full Dataset**: Remove the `SAMPLE_SIZE` limit to process all rows
2. **Tune Parameters**: Adjust `max_keywords`, temperature, and other model parameters
3. **Try Different Models**: Experiment with other WatsonX.ai models
4. **Post-Processing**: Filter, deduplicate, or enhance extracted keywords/entities
5. **Integration**: Use extracted data for categorization, search, or analytics

## Additional Resources

- [IBM WatsonX.ai Documentation](https://www.ibm.com/docs/en/watsonx-as-a-service)
- [IBM Cloud Object Storage Documentation](https://cloud.ibm.com/docs/cloud-object-storage)
- Project README: `../README.md`
- Credentials Setup: `../IBM_CREDENTIALS_SETUP.md`